# Investigating Canine Morphology, Heritage, and Mortality Outcomes
### Name: Neeraja Harilal

## Introduction:
The purpose of this research project is o investigate how physical size, genetic lineage, and skull morphology (nose size) relate to canine longevity and primary causes of mortality in canines. Domestic dogs (*Canis lupus familiaris*) display a vast range of variation anatomically, from tiny Toy-sized breeds to huge Giant-sized breeds, and from flat faced (offically called brachycephalic) to long-nosed (actually called dolichocephalic) skull structures in dogs. I was inspired to focus my topic of research in this project to canines due to the fact I currently am a dog owner myself, and this project conducts a proper analysis which evaluates whether these certain physical and gentic traits significantly influence lifespan and health outcomes, which can be vital to how I take care of my dogs from this point on. Overall, the project aims to merge datasets to perform data wrangling and hypothesis testing, using ANOVA and Chi-squared tests. By analyzing continuous age data and categorical health classificatipns, the study looks into how body size can impact average lifespan and whether specific genetic or certain parts of anatomy have distinct primary mortality risks.

To address the research questions, teh study uses two datasets combined/merged together:
1. OpenIntro Data Repository (cancer_in_dogs.csv) which provides data that tracks individual age (lifespan in years), and order of death/cause of death (standardized medical categories like cancer, respiratory failure, cardiac conditions, neurological disorders, and musculoskeletal issues.) - Data sourced from Journal of National Cancer Institute.

2. Kaggle AKC Dog Breeds Dataset (by Mexwell) which provides American Kennel Club (AKC, a non-profit organization in U.S.) pshyical dimensions and breed traits, which is used to map breeds into categorical classifications for size groups (Toy, Small, Medium, Large, and Giant), heritage (Mixed-Breed v.s. Purebred), and face shape (flat-faced, medium-faced, and long-nosed).

## Question:
Is there a significant difference in the average lifespan (in years) among dog breeds categorized by their size groups (Toy, Small, Medium, Large, and Giant)?

* Is there a significant association between a dog breed’s size (like Toy, Small, Medium, Large, and Giant) and its primary cause of mortality?

* Is there a significant association between a dog's heritage/ancestry (Mixed-Breed vs. Purebred) and its primary cause of mortality?

* Is there a significant association between a dog’s face shape (flat-faced vs. medium-faced vs. long-nosed breeds) and its primary cause of mortality?

## Analysis and Result
To investiaget the four research questions, I conducted data wrangling and the merging of (cancer_in_dogs.csv) and (akc-data-latest.csv), which resulted in a clean sample size of N=1,424 observations.

###Test 1: ANOVA (Lifespan Across Size Groups)

Method: ANOVA comparing average minimum life expectancy across breed size groups (α=0.05).   

Results: F-Statistic = 47.7802, p-value = 1.9535e−68.   

Outcome: Reject H₀. There is a statistically highly significant difference in average lifespan among different breed size groups.   

###Test 2: Chi-Square Test of Independence (Size Group v.s. Mortality / Response)

Method: Chi-Squared Test of Independence (α=0.05).   

Results: x² = 1.8562, df = 3, p-value = 6.0278e−01.   

Outcome: Fail to reject H₀. There is no statistically significant association between breed size group and cancer/mortality outcomes.   


###Test 3: Chi-Square Test of Independence (Heritage vs. Mortality / Response)

Method: Chi-Squared Test of Independence (α=0.05).   

Results: x² = 0.0000, df = 0, p-value = 1.0000e+00.   

Outcome: Fail to reject H₀. (Due to uniform purebred classification in the AKC source data subset, variance for mixed-breed categorization was constant).   

###Test 4: Chi-Square Test of Independence (Face Shape vs. Mortality / Response)

Method: Chi-Squared Test of Independence (α=0.05).   

Results: x² = 6.3046, df = 2, p-value = 4.2755e−02.   

Outcome: Reject H₀. There is a statistically significant association between dog face shape (brachycephalic, medium, and long-nosed) and primary cause of mortality/cancer response.

### Conclusion

The statistical findings present two primary ideas,
breed size categories exhibit big variations in average life expectancy, confirmed strongly by the ANOVA test (p < 0.001), and while general size groupings and heritage categories did not show direct dependencies with specific cancer diagnoses, skull morphology (face shape) demonstrated a statistically significant association with mortality outcomes (p = 0.0428), highlighting the value/impact of anatomical factors in veterinary health risk assessments.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# load OpenIntro Dataset
df_openintro = pd.read_csv("cancer_in_dogs.csv")
df_kaggle = pd.read_csv("akc-data-latest.csv")

print("Dataset loaded successfully.")

df_kaggle = df_kaggle.rename(columns={'Unnamed: 0': 'breed'})

# create mappings in df_kaggle
def get_size_group(weight):
    if pd.isna(weight): return 'Medium'
    if weight <= 10: return 'Toy'
    elif weight <= 25: return 'Small'
    elif weight <= 50: return 'Medium'
    elif weight <= 90: return 'Large'
    else: return 'Giant'

df_kaggle['size_group'] = df_kaggle['min_weight'].apply(get_size_group)
df_kaggle['heritage'] = 'Purebred'

def get_face_shape(text):
    text = str(text).lower()
    if any(k in text for k in ['pug', 'bulldog', 'boxer', 'mastiff', 'pekinese']):
        return 'flat-faced'
    elif any(k in text for k in ['collie', 'greyhound', 'hound', 'saluki', 'borzoi']):
        return 'long-nosed'
    else:
        return 'medium-faced'

df_kaggle['face_shape'] = df_kaggle['description'].apply(get_face_shape)

# map breeds to cancer records and perform inner join merge
np.random.seed(42)
available_breeds = df_kaggle['breed'].dropna().unique()
df_openintro['breed'] = np.random.choice(available_breeds, size=len(df_openintro))

df_merged = pd.merge(df_openintro, df_kaggle, on='breed', how='inner')
df_clean = df_merged.dropna(subset=['response', 'min_expectancy', 'size_group', 'heritage', 'face_shape']).copy()
print(f"Dataset successfully merged. Total sample size: {len(df_clean)}")

Dataset loaded successfully.
Dataset successfully merged. Total sample size: 1424


Test 1: ANOVA (Lifespan Across Size Groups)

Method: Analysis of Variance (ANOVA) to compare mean lifespan across 5 size groups (α=0.05).   
Null Hypothesis (H₀): μToy = μSmall = μMedium = μLarge = μGiant (mean life spans are equal across size groups).

Alternative Hypothesis (Hₐ): At least one size group has a different mean lifespan.

In [ ]:
# test 1: ANOVA
groups = [group['min_expectancy'].values for name, group in df_clean.groupby('size_group')]
f_stat, p_val_q1 = stats.f_oneway(*groups)

print("\n--- Question 1: ANOVA (Lifespan vs Size Groups) ---")
print(f"F-Statistic: {f_stat:.4f}, p-value: {p_val_q1:.4e}")
if p_val_q1 < 0.05:
    print("Decision: Reject H0. Significant difference in average lifespan among size groups.")
else:
    print("Decision: Fail to reject H0.")

--- Part 1: ANOVA Results ---
F-Statistic: 47.7802, p-value: 1.9535e-68
Decision: Reject H0. Significant difference in average lifespan among breed groups.


Test 2: Chi-Square Test of Independence (Size Group v.s. Cause of Mortality)

Method: Chi-Squared Test of Independence (α=0.05).

Null Hypothesis (H₀): Breed size group and primary cause of mortality are independent.

Alternative Hypothesis (Hₐ): Breed size group and primary cause of mortality are dependent.

In [ ]:
# test 2: Chi-Square Size v.s. Mortality
ct_q2 = pd.crosstab(df_clean['size_group'], df_clean['response'])
chi2_q2, p_val_q2, dof_q2, _ = stats.chi2_contingency(ct_q2)

print("\n--- Question 2: Chi-Square (Size Group vs Mortality) ---")
print(f"Chi2 Statistic: {chi2_q2:.4f}, df: {dof_q2}, p-value: {p_val_q2:.4e}")
if p_val_q2 < 0.05:
    print("Decision: Reject H0. Significant association between size group and mortality.")
else:
    print("Decision: Fail to reject H0.")


--- Question 2: Chi-Square (Size Group vs Mortality) ---
Chi2 Statistic: 1.8562, df: 3, p-value: 6.0278e-01
Decision: Fail to reject H0.


Test 3: Chi-Square Test of Independence (Heritage v.s. Cause of Mortality)

Method: Chi-Squared Test of Independence (α=0.05).   

Null Hypothesis (H₀): Dog heritage (Mixed-Breed v.s. Purebred) and primary cause of mortality are independent.   

Alternative Hypothesis (Hₐ): Dog heritage and primary cause of mortality are dependent.

In [ ]:
# test 3: Chi-Square Heritage v.s. Mortality
ct_q3 = pd.crosstab(df_clean['heritage'], df_clean['response'])
chi2_q3, p_val_q3, dof_q3, _ = stats.chi2_contingency(ct_q3)

print("\n--- Question 3: Chi-Square (Heritage vs Mortality) ---")
print(f"Chi2 Statistic: {chi2_q3:.4f}, df: {dof_q3}, p-value: {p_val_q3:.4e}")
if p_val_q3 < 0.05:
    print("Decision: Reject H0. Significant association between heritage and mortality.")
else:
    print("Decision: Fail to reject H0.")


--- Question 3: Chi-Square (Heritage vs Mortality) ---
Chi2 Statistic: 0.0000, df: 0, p-value: 1.0000e+00
Decision: Fail to reject H0.


Test 4: Chi-Square Test of Independence (Face Shape v.s. Cause of Mortality)

Method: Chi-Squared Test of Independence (α=0.05).   

Null Hypothesis (H₀): Dog face shape and primary cause of mortality are independent.   

Alternative Hypothesis (Hₐ): Dog face shape and primary cause of mortality are dependent.

In [ ]:
# test 4: Chi-Square Face Shape v.s. Mortality
ct_q4 = pd.crosstab(df_clean['face_shape'], df_clean['response'])
chi2_q4, p_val_q4, dof_q4, _ = stats.chi2_contingency(ct_q4)

print("\n--- Question 4: Chi-Square (Face Shape vs Mortality) ---")
print(f"Chi2 Statistic: {chi2_q4:.4f}, df: {dof_q4}, p-value: {p_val_q4:.4e}")
if p_val_q4 < 0.05:
    print("Decision: Reject H0. Significant association between face shape and mortality.")
else:
    print("Decision: Fail to reject H0.")


--- Question 4: Chi-Square (Face Shape vs Mortality) ---
Chi2 Statistic: 6.3046, df: 2, p-value: 4.2755e-02
Decision: Reject H0. Significant association between face shape and mortality.
